# SSTW B6 Sampling-Time Constraint Colab - Wan2.1 主线 入口

该 Notebook 只作为 Colab L4 入口使用。正式 records、tables、reports、artifacts 和 package manifest 由仓库模块生成。

默认落盘根目录: `/content/drive/MyDrive/SSTW`。当前阶段是 `sampling_time_constraint_colab_probe`, 不等同于最终 B6 submission freeze claim。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, json
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SSTW'
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 2. 获取仓库
REPO_URL = 'https://github.com/RICHAAARC/SSTW.git'
REPO_DIR = '/content/SSTW'

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"


In [ ]:
# 3. 安装 Colab GPU 运行依赖
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf


In [ ]:
# 4. 可选 Hugging Face 认证
# 公开模型通常可以匿名下载, 但建议提供 HF_TOKEN 以避免限流。token 不写入正式 records。
import os
try:
    from huggingface_hub import login
except Exception as exc:
    raise RuntimeError('huggingface_hub 未安装或无法导入, 请重新执行依赖安装单元') from exc

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 初始化 Drive 归档布局, 并切换本地阶段 workspace
from paper_workflow.notebook_utils import sampling_time_constraint_workflow as constraint_workflow
from paper_workflow.colab_utils.stage_package_sync import prepare_colab_stage_layout, publish_colab_stage_package

os.environ.setdefault('SSTW_COLAB_STAGE_IO_MODE', 'local_zip')
drive_layout = constraint_workflow.ensure_drive_layout(DRIVE_PROJECT_ROOT)
layout = prepare_colab_stage_layout(
    drive_layout,
    notebook_role='sampling_time_constraint',
)
print(json.dumps({
    'drive_layout': drive_layout,
    'active_local_layout': layout,
}, ensure_ascii=False, indent=2))


In [ ]:
# 6. 构造 prompt / seed 数据集
cmd = constraint_workflow.build_prompt_suite_command(layout)
print(' '.join(cmd))
result = constraint_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 7. 配置 B6 Colab probe
PROFILE = 'recommended'  # 已通过 smoke, 当前切换为 recommended 以扩展 prompt / seed 覆盖。
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
SEMANTIC_MODEL_ID = 'openai/clip-vit-base-patch32'
SEMANTIC_FRAME_LIMIT = 8
INCLUDE_VIDEOS_IN_PACKAGE = True


In [ ]:
# 8. 执行 B6 sampling-time constraint Colab probe
cmd = constraint_workflow.build_sampling_constraint_colab_runtime_command(layout, PROFILE, MODEL_ID)
print(' '.join(cmd))
result = constraint_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 9. 执行真实视频质量、运动与 CLIP 语义 metric
cmd = constraint_workflow.build_formal_metric_command(layout, semantic_model_id=SEMANTIC_MODEL_ID, semantic_frame_limit=SEMANTIC_FRAME_LIMIT)
print(' '.join(cmd))
result = constraint_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 10. 执行 B6 后处理
cmd = constraint_workflow.build_postprocess_command(layout)
print(' '.join(cmd))
result = constraint_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 11. 检查 B6 Colab 结果证据状态
cmd = constraint_workflow.build_result_check_command(layout)
print(' '.join(cmd))
result = constraint_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 12. 运行 quick tests 与 harness 审计
!pytest -q
!python tools/harness/run_all_audits.py


In [ ]:
# 13. 打包到 Google Drive packages/sampling_time_constraint
cmd = constraint_workflow.build_drive_packaging_command(layout, include_videos=INCLUDE_VIDEOS_IN_PACKAGE)
print(' '.join(cmd))
result = constraint_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)

stage_package = publish_colab_stage_package(
    layout,
    notebook_role='sampling_time_constraint',
    include_videos=INCLUDE_VIDEOS_IN_PACKAGE,
)
print(json.dumps(stage_package, ensure_ascii=False, indent=2))


In [ ]:
# 14. 显示可下载审阅文件
package_dir = Path(layout['drive_package_dir'])
print('package_dir =', package_dir)
for path in sorted(package_dir.glob('*')):
    print(path)
